# Hosting Strands Agents with OpenAI models in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to host your existing agent, using Amazon Bedrock AgentCore Runtime. 

We will focus on a Strands Agents with OpenAI model example. For Strands Agents with Amazon Bedrock model check [here](../01-strands-with-bedrock-model) and 
for LangGraph with Amazon Bedrock model check [here](../02-langgraph-with-bedrock-model)


### Tutorial Details

| Information         | Details                                                                  |
|:--------------------|:-------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                           |
| Agent type          | Single                                                                   |
| Agentic Framework   | Strands Agents                                                           |
| LLM model           | GPT 4.1 mini                                                             |
| Tutorial components | Hosting agent on AgentCore Runtime. Using Strands Agent and OpenAI Model |
| Tutorial vertical   | Cross-vertical                                                           |
| Example complexity  | Easy                                                                     |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                             |

### Tutorial Architecture

In this tutorial we will describe how to deploy an existing agent to AgentCore runtime. 

For demonstration purposes, we will  use a Strands Agent using Amazon Bedrock models

In our example we will use a very simple agent with two tools: `get_weather` and `get_time`. 

<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="60%"/>
</div>

### Tutorial Key Features

* Hosting Agents on Amazon Bedrock AgentCore Runtime
* Using OpenAI models
* Using Strands Agents


## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker running

## Creating your agents and experimenting locally

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

For production agentic applications we will need to decouple the agent creation process from the agent invocation one. With AgentCore Runtime, we will decorate the invocation part of our agent with the `@app.entrypoint` decorator and have it as the entry point for our runtime. Let's first look how each agent is developed during the experimentation phase.

The architecture here will look as following:

<div style="text-align:left">
    <img src="images/architecture_local.png" width="60%"/>
</div>

In [ ]:
%%writefile strands_agents_openai.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from strands.models.litellm import LiteLLMModel
import os
import boto3

ssm = boto3.client('ssm', region_name='ap-southeast-1')

parameter = ssm.get_parameter(
    Name='/llm-provider/openrouter/api-key',
    WithDecryption=True
)

OPENROUTER_API_KEY = parameter['Parameter']['Value'] #aws ssm get-parameters --name /llm-provider/openrouter/api-key --with-decryption

# Set environment for LiteLLM
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENROUTER_API_BASE"] = "https://openrouter.ai/api/v1"

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"

model = "openrouter/qwen/qwen3-235b-a22b-2507"
litellm_model = LiteLLMModel(
    model_id=model, params={"max_tokens": 1000, "temperature": 0.7}
)


agent = Agent(
    model=litellm_model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

def strands_agent_open_ai(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = strands_agent_open_ai(json.loads(args.payload))
    print(response)

Overwriting strands_agents_openai.py


#### Invoking local agent

In [24]:
!uv run strands_agents_openai.py '{"prompt": "What is the weather now?"}'


Tool #1: weather
The current weather is sunny! It's a beautiful day outside.The current weather is sunny! It's a beautiful day outside.


## Preparing your agent for deployment on AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to:
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

### Strands Agents with OpenAI model
Let's start with our Strands Agent using the GPT 4.1 mini model. All the others will work exactly the same.

In [ ]:
%%writefile strands_agents_openai.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from strands.models.litellm import LiteLLMModel
import os
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import boto3

app = BedrockAgentCoreApp()

ssm = boto3.client('ssm', region_name='ap-southeast-1')

parameter = ssm.get_parameter(
    Name='/llm-provider/openrouter/api-key',
    WithDecryption=True
)

OPENROUTER_API_KEY = parameter['Parameter']['Value'] #aws ssm get-parameters --name /llm-provider/openrouter/api-key --with-decryption

# Set environment for LiteLLM
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENROUTER_API_BASE"] = "https://openrouter.ai/api/v1"

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"

model = "openrouter/qwen/qwen3-30b-a3b-instruct-2507"
litellm_model = LiteLLMModel(
    model_id=model, params={"max_tokens": 32000, "temperature": 0.3}
)


agent = Agent(
    model=litellm_model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)


@app.entrypoint
def strands_agent_open_ai(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

Overwriting strands_agents_openai.py


## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [49]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import boto3
import json
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()

agent_name = "strands_openai_getting_started"
response = agentcore_runtime.configure(
    entrypoint="strands_agents_openai.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
)
response

Entrypoint parsed: file=/Users/ode/Documents/Workspace/work/the-smol-lab/smol-to-prod-samples/strands-with-openai-model/strands_agents_openai.py, bedrock_agentcore_name=strands_agents_openai
Configuring BedrockAgentCore agent: strands_openai_getting_started
Generated Dockerfile: /Users/ode/Documents/Workspace/work/the-smol-lab/smol-to-prod-samples/strands-with-openai-model/Dockerfile
Generated .dockerignore: /Users/ode/Documents/Workspace/work/the-smol-lab/smol-to-prod-samples/strands-with-openai-model/.dockerignore
Keeping 'strands_openai_getting_started' as default agent
Bedrock AgentCore configured: /Users/ode/Documents/Workspace/work/the-smol-lab/smol-to-prod-samples/strands-with-openai-model/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/Users/ode/Documents/Workspace/work/the-smol-lab/smol-to-prod-samples/strands-with-openai-model/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/ode/Documents/Workspace/work/the-smol-lab/smol-to-prod-samples/strands-with-openai-model/Dockerfile'), dockerignore_path=PosixPath('/Users/ode/Documents/Workspace/work/the-smol-lab/smol-to-prod-samples/strands-with-openai-model/.dockerignore'), runtime='Docker', region='ap-southeast-1', account_id='733653166178', execution_role=None, ecr_repository=None, auto_create_ecr=True)

### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [50]:
launch_result = agentcore_runtime.launch()

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'strands_openai_getting_started' to account 733653166178 (ap-southeast-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: strands_openai_getting_started
✅ ECR repository available: 733653166178.dkr.ecr.ap-southeast-1.amazonaws.com/bedrock-agentcore-strands_openai_getting_started
Getting or creating execution role for agent: strands_openai_getting_started
Using AWS region: ap-southeast-1, account ID: 733653166178
Role name: AmazonBedrockAgentCoreSDKRuntime-ap-southeast-1-aeccffb574


✅ Reusing existing ECR repository: 733653166178.dkr.ecr.ap-southeast-1.amazonaws.com/bedrock-agentcore-strands_openai_getting_started


✅ Reusing existing execution role: arn:aws:iam::733653166178:role/AmazonBedrockAgentCoreSDKRuntime-ap-southeast-1-aeccffb574
✅ Execution role available: arn:aws:iam::733653166178:role/AmazonBedrockAgentCoreSDKRuntime-ap-southeast-1-aeccffb574
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: strands_openai_getting_started
Role name: AmazonBedrockAgentCoreSDKCodeBuild-ap-southeast-1-aeccffb574
Reusing existing CodeBuild execution role: arn:aws:iam::733653166178:role/AmazonBedrockAgentCoreSDKCodeBuild-ap-southeast-1-aeccffb574
Using .dockerignore with 44 patterns
Uploaded source to S3: strands_openai_getting_started/source.zip
Updated CodeBuild project: bedrock-agentcore-strands_openai_getting_started-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 8.6s
🔄 DOWNLO

### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [51]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

Retrieved Bedrock AgentCore status for: strands_openai_getting_started


'READY'

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

<div style="text-align:left">
    <img src="images/invoke.png" width=85%"/>
</div>

In [52]:
invoke_response = agentcore_runtime.invoke({"prompt": "Hi, what can you do?"})
invoke_response

{'ResponseMetadata': {'RequestId': '0ca408d6-35c4-4a24-b061-14f425ac0bd3',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Mon, 10 Nov 2025 12:22:18 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': '0ca408d6-35c4-4a24-b061-14f425ac0bd3',
   'baggage': 'Self=1-6911d8eb-3b4eedd92bab25a0475907e1,session.id=569ce2b3-d24e-42e3-a54e-05b8bf86e5e7',
   'x-amzn-bedrock-agentcore-runtime-session-id': '569ce2b3-d24e-42e3-a54e-05b8bf86e5e7',
   'x-amzn-trace-id': 'Root=1-6911d8eb-0446796b59cbff2921ae8fd1;Parent=97233d2e940a739f;Sampled=1;Self=1-6911d8eb-3b4eedd92bab25a0475907e1'},
  'RetryAttempts': 0},
 'runtimeSessionId': '569ce2b3-d24e-42e3-a54e-05b8bf86e5e7',
 'traceId': 'Root=1-6911d8eb-0446796b59cbff2921ae8fd1;Parent=97233d2e940a739f;Sampled=1;Self=1-6911d8eb-3b4eedd92bab25a0475907e1',
 'baggage': 'Self=1-6911d8eb-3b4eedd92bab25a0475907e1,session.id=569ce2b3-d24e-42e3-a54e-05b8bf86e5e7',
 'contentType': 

### Processing invocation results

We can now process our invocation results to include it in an application

In [53]:
from IPython.display import Markdown, display
import json

response_text = invoke_response['response'][0]
display(Markdown(response_text))

Hi! I can help you with a variety of tasks:

1. **Math calculations** - Simple arithmetic, algebra, calculus (derivatives, integrals), and more.
2. **Equation solving** - Find solutions to equations or systems of equations.
3. **Weather information** - Check current weather conditions.
4. **Scientific notation** - Work with large or small numbers in scientific format.
5. **Matrix operations** - Perform calculations with matrices.

Just let me know what you need! 😊

In [58]:
from IPython.display import Markdown, display
import json


invoke_response = agentcore_runtime.invoke({"prompt": "Compute 124152+123450*256/8"})
response_text = invoke_response['response'][0]
display(Markdown(response_text))

The result of the calculation $ 124152 + \frac{123450 \times 256}{8} $ is **4,074,552**.

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "What is the point of life,?"})
response_text = invoke_response['response'][0]
display(Markdown(response_text))

That's a profound and deeply philosophical question! The "point of life" is something humans have pondered for centuries, and there's no single answer that fits everyone. Here are a few perspectives:

- **Philosophical**: Some believe life's purpose is to seek meaning, pursue truth, and grow as individuals.
- **Religious/Spiritual**: Many faiths suggest life's purpose is to serve a higher power, follow moral principles, or achieve spiritual enlightenment.
- **Scientific**: From a biological standpoint, life's "purpose" might be to survive, reproduce, and pass on genes.
- **Personal**: For many, the point of life is found in relationships, creativity, learning, helping others, or contributing to something greater.

Ultimately, the meaning of life may not be something you find—it could be something you create through your choices, values, and experiences.

What do *you* think gives your life meaning? 😊

In [63]:
invoke_response = agentcore_runtime.invoke({"prompt": "Teach me economics in a single paragraph."})
response_text = invoke_response['response'][0]
display(Markdown(response_text))

Economics is the study of how individuals, businesses, governments, and societies allocate scarce resources to satisfy unlimited wants, focusing on the choices people make when faced with trade-offs—such as producing more of one good versus another, or saving versus spending. At its core, it examines supply and demand, where prices are determined by the interaction of what producers are willing to supply and what consumers are willing to buy, leading to market equilibrium. It also explores broader concepts like opportunity cost (the value of the next best alternative), incentives, scarcity, and the role of institutions in shaping economic behavior. Economics is divided into microeconomics (individual and firm-level decisions) and macroeconomics (national and global issues like inflation, unemployment, and economic growth), and it uses models, data, and critical thinking to understand and predict how economies function and evolve over time.

### Invoking AgentCore Runtime with boto3

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [60]:
# import boto3
# agent_arn = launch_result.agent_arn
# agentcore_client = boto3.client(
#     'bedrock-agentcore',
#     region_name=region
# )

# boto3_response = agentcore_client.invoke_agent_runtime(
#     agentRuntimeArn=agent_arn,
#     qualifier="DEFAULT",
#     payload=json.dumps({"prompt": "What is the point of life?"})
# )
# if "text/event-stream" in boto3_response.get("contentType", ""):
#     content = []
#     for line in boto3_response["response"].iter_lines(chunk_size=1):
#         if line:
#             line = line.decode("utf-8")
#             if line.startswith("data: "):
#                 line = line[6:]
#                 print(line)
#                 content.append(line)
#     display(Markdown("\n".join(content)))
# else:
#     try:
#         events = []
#         for event in boto3_response.get("response", []):
#             events.append(event)
#     except Exception as e:
#         events = [f"Error reading EventStream: {e}"]
#     display(Markdown(json.loads(events[0].decode("utf-8"))))

## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [ ]:
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region
    
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

# Congratulations!